# Training Loop Example

In this notebook, we implement a training loop for a basic transformer model in pyTorch on the tiny shakespeare data set, as a hommage to nanoGPT from Andrej Karpathy's brilliant [Neural Networks: Zero to Hero](https://www.youtube.com/playlist?list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ) series.

## Imports and Setup

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [8]:
from typing import List

In [9]:
from adam import Adam

In [10]:
file_path = "../data/input.txt"

with open(file_path, "r") as f:
    dataset = list(f.read())

## Dataset Preparation

In [11]:
split = round(len(dataset)*0.9)
data_train = dataset[:split]
data_val = dataset[split+1:]

chars = sorted(set(dataset))
ctoi = {c:i for i, c in enumerate(chars)}
itoc = {ctoi[c]:c for c in ctoi.keys()}

data_train = [ctoi[c] for c in data_train]
data_val = [ctoi[c] for c in data_val]

In [12]:
encode = lambda s: [ctoi[c] for c in s]
decode = lambda s: "".join([itoc[i] for i in s])

In [26]:
def get_batch(data, block_size):
    batch_idx = np.random.randint(0, len(data) - block_size - 1)
    xbatch = data[batch_idx : batch_idx + block_size]
    ybatch = data[batch_idx + 1 : batch_idx + block_size + 1]

    return xbatch, ybatch

## Model Initialization

In [28]:
class Model(nn.Module):

    def __init__(self, block_size: int, vocab_size: int, d_model: int, nhead: int, num_layers: int):
        super().__init__()
        self.block_size = block_size

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.positional_embedding = nn.Embedding(block_size, d_model)
        self.causal_mask = torch.triu(torch.ones(block_size, block_size), diagonal=1).bool()

        self.encoding_layer = nn.TransformerEncoderLayer(d_model, nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.encoding_layer, num_layers)

        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, inputs: List):
        embeddings = self.token_embedding(inputs)
        pos_embeddings = self.positional_embedding(range(self.block_size))
        embeddings += pos_embeddings

        transformer_outputs = self.transformer(embeddings, self.causal_mask)
        logits = self.lm_head(transformer_outputs)
        return logits

In [19]:
block_size = 32
vocab_size = len(chars)
d_model = 15
nhead = 3
num_layers = 5

## Training Loop

In [23]:
model = Model(block_size, vocab_size, d_model, nhead, num_layers)

In [27]:
num_iters = 1000
params = model.parameters()

optim = Adam(params)

In [ ]:
for _ in range(num_iters):
    xtrain, ytrain = get_batch(data_train, block_size)
    logits = model(xtrain)
    loss = F.cross_entropy(logits, ytrain)

    optim.zero_grad()
    loss.backward()
    optim.step()